# PaddleOCR OCR Benchmark (Self-Contained for Colab)

This notebook runs the full PaddleOCR benchmark on the Nepali Pixel dataset.
It is completely self-contained (no local imports required) so you can run it directly in Google Colab.
Ensure you have enabled a GPU runtime in Colab for optimal performance.

In [ ]:
# 1. Install System Dependencies and Python Libraries
!python -m pip install paddlepaddle-gpu paddleocr datasets Pillow

In [ ]:
# 2. Imports
import os
import re
import sys
import time
import json
import string
import collections
import statistics
import concurrent.futures
from datetime import datetime, timezone
from pathlib import Path
from difflib import SequenceMatcher
from typing import Any, Dict, Iterator, Tuple, Sequence

from paddleocr import PaddleOCR
from datasets import load_dataset
from PIL.Image import Image
import numpy as np

In [ ]:
# 3. Scoring Metrics Definition
def levenshtein_distance(ref: Sequence, hyp: Sequence) -> int:
    m, n = len(ref), len(hyp)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
        
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1, dp[i - 1][j - 1] + cost)
    return dp[m][n]

def character_error_rate(reference: str, hypothesis: str) -> float:
    if not reference: return 1.0 if hypothesis else 0.0
    return levenshtein_distance(reference, hypothesis) / len(reference)

def word_error_rate(reference: str, hypothesis: str) -> float:
    ref_words = reference.split()
    hyp_words = hypothesis.split()
    if not ref_words: return 1.0 if hyp_words else 0.0
    return levenshtein_distance(ref_words, hyp_words) / len(ref_words)

def exact_match(reference: str, hypothesis: str) -> bool:
    return reference == hypothesis

def normalize_text(text: str) -> str:
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalized_exact_match(reference: str, hypothesis: str) -> bool:
    return normalize_text(reference) == normalize_text(hypothesis)

def length_ratio(reference: str, hypothesis: str) -> float:
    if not reference: return 1.0 if not hypothesis else float('inf')
    return len(hypothesis) / len(reference)

def char_f1_score(reference: str, hypothesis: str) -> float:
    if not reference and not hypothesis: return 1.0
    if not reference or not hypothesis: return 0.0
    sm = SequenceMatcher(None, reference, hypothesis)
    match_size = sum(m.size for m in sm.get_matching_blocks())
    precision = match_size / len(hypothesis) if len(hypothesis) > 0 else 0
    recall = match_size / len(reference) if len(reference) > 0 else 0
    if precision + recall == 0: return 0.0
    return 2 * (precision * recall) / (precision + recall)

def compute_metrics(reference: str, hypothesis: str) -> dict:
    return {
        "cer": character_error_rate(reference, hypothesis),
        "wer": word_error_rate(reference, hypothesis),
        "exact_match": exact_match(reference, hypothesis),
        "normalized_exact_match": normalized_exact_match(reference, hypothesis),
        "char_f1": char_f1_score(reference, hypothesis),
        "length_ratio": length_ratio(reference, hypothesis)
    }

In [ ]:
# 4. Adapter & Dataset Setup
class PaddleOCRAdapter:
    def __init__(self, lang: str = "devanagari"):
        # PaddleOCR introduced 'devanagari' for Indic languages
        self.ocr = PaddleOCR(use_angle_cls=True, lang=lang, use_gpu=True, show_log=False)
        
    def evaluate_sample(self, image: Image) -> str:
        img_np = np.array(image.convert('RGB'))
        results = self.ocr.ocr(img_np, cls=True)
        if not results or not results[0]:
            return ""
        text_blocks = [res[1][0] for res in results[0]]
        return " ".join(text_blocks)

def load_nepali_pixel_dataset(split: str = "train") -> Iterator[Tuple[str, Image, str, Dict[str, Any]]]:
    dataset = load_dataset("himalaya-ai/nepalipixel-synthetic-ocr-benchmark", split=split)
    for row in dataset:
        sample_id = str(row.get("id", ""))
        image = row.get("image")
        ground_truth = row.get("ground_truth", row.get("text", ""))
        
        metadata = {
            "id": sample_id,
            "level": row.get("level", ""),
            "font_name": row.get("font_name", ""),
            "intensity": row.get("intensity", ""),
            "augmentations": row.get("augmentations", [])
        }
        yield sample_id, image, ground_truth, metadata

In [ ]:
# 5. Configuration
LIMIT = None # Set to an integer (e.g., 50) for a quick smoke-test

def timestamp() -> str:
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

output_dir = Path(f"results/paddleocr_run_notebook_{timestamp()}")
model_name = "paddleocr"
model_dir = output_dir / "models" / model_name
model_dir.mkdir(parents=True, exist_ok=True)

samples_path = model_dir / "samples.jsonl"
progress_path = model_dir / "progress.json"

print(f"Results and progress will be saved to: {output_dir}")

In [ ]:
# 6. Execution Helpers
def evaluate_one(item, adapter):
    sample_id, image, ground_truth, metadata = item
    result = {
        "sample_id": sample_id,
        "ground_truth": ground_truth,
        "metadata": metadata,
        "model": "paddleocr"
    }
    
    started = time.perf_counter()
    try:
        response_text = adapter.evaluate_sample(image)
        latency = time.perf_counter() - started
        metrics = compute_metrics(ground_truth, response_text)
        result.update({
            "ok": True,
            "latency_sec": latency,
            "response": response_text,
            "metrics": metrics,
        })
    except Exception as exc:
        result.update({
            "ok": False,
            "latency_sec": None,
            "response": "",
            "error": str(exc),
            "metrics": {"cer": 1.0, "wer": 1.0, "exact_match": False, "normalized_exact_match": False, "char_f1": 0.0, "length_ratio": 0.0}
        })
    return result

def append_jsonl(handle, row) -> None:
    handle.write(json.dumps(row, ensure_ascii=False))
    handle.write("\n")
    handle.flush()

In [ ]:
# 7. Run Benchmark
print("Initializing Dataset & Adapter...")
dataset_gen = load_nepali_pixel_dataset(split="train")
adapter = PaddleOCRAdapter()

samples = []
started_at = datetime.now(timezone.utc).isoformat()
completed = 0
errored = 0

print("Starting sequential evaluation to conserve memory...", flush=True)
with samples_path.open("w", encoding="utf-8") as samples_handle:
    for i, item in enumerate(dataset_gen):
        if LIMIT and i >= LIMIT:
            break
        
        # Execute
        sample = evaluate_one(item, adapter)
        samples.append(sample)
        append_jsonl(samples_handle, sample)
        
        completed += 1
        if not sample["ok"]:
            errored += 1
            
        if completed % 25 == 0:
            progress = {
                "model": model_name,
                "completed_completions": completed,
                "errored_completions": errored,
                "updated_at": datetime.now(timezone.utc).isoformat(),
            }
            with progress_path.open("w") as f:
                json.dump(progress, f)
            print(f"\rProgress: {completed} completions processed (Errors: {errored})", end="", flush=True)

print(f"\nBenchmark finished! Processed {completed} items.")

In [ ]:
# 8. Summarize Results
valid_samples = [s for s in samples if s["ok"]]
errored_completions = len(samples) - len(valid_samples)

if samples and (errored_completions / len(samples)) > 0.5:
    error_counts = collections.Counter(s["error"] for s in samples if not s["ok"])
    print(f"\n✗ Run ABORTED: {errored_completions}/{len(samples)} samples errored.")
    for msg, count in error_counts.most_common():
        print(f"  [{count}x] {msg}")
else:
    summary = {
        "model": model_name,
        "started_at": started_at,
        "completed_at": datetime.now(timezone.utc).isoformat(),
        "examples": len(samples),
        "errored_completions": errored_completions,
        "avg_cer": statistics.fmean(s["metrics"]["cer"] for s in valid_samples) if valid_samples else None,
        "avg_wer": statistics.fmean(s["metrics"]["wer"] for s in valid_samples) if valid_samples else None,
        "exact_match_rate": statistics.fmean(float(s["metrics"]["exact_match"]) for s in valid_samples) if valid_samples else None,
        "normalized_exact_match_rate": statistics.fmean(float(s["metrics"]["normalized_exact_match"]) for s in valid_samples) if valid_samples else None,
        "avg_char_f1": statistics.fmean(s["metrics"]["char_f1"] for s in valid_samples) if valid_samples else None,
        "avg_length_ratio": statistics.fmean(s["metrics"]["length_ratio"] for s in valid_samples) if valid_samples else None,
        "avg_latency_sec": statistics.fmean(s["latency_sec"] for s in valid_samples) if valid_samples else None,
    }

    with (model_dir / "summary.json").open("w") as f:
        json.dump(summary, f, indent=2)
        
    run_summary = {
        "run_id": output_dir.name,
        "models": [model_name],
        "completed_at": datetime.now(timezone.utc).isoformat(),
        "model_summaries": [summary]
    }
    with (output_dir / "summary.json").open("w") as f:
        json.dump(run_summary, f, indent=2)

    print("\n=== Final Benchmark Summary ===")
    for k, v in summary.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")